# Retrieval Evaluation

This notebook contains synthetic retrieval outputs for a mocked RAG pipeline.

Your task: decide what to measure, implement an evaluation, and explain what the outputs imply. You can use pandas, scikit-learn, or any other approach you think is appropriate.

## Dataset

The notebook loads four pandas DataFrames:

- `document_catalog_df`: available documents.
- `queries_df`: evaluation queries.
- `relevance_df`: expected relevant documents for each query.
- `retrieval_df`: ranked retrieval outputs, with three runs per query.

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 120)

In [ ]:
document_catalog_df = pd.DataFrame(
    [
        {"doc_id": "policy-auth", "title": "Enterprise authentication", "topic": "identity"},
        {"doc_id": "policy-privacy", "title": "Data privacy", "topic": "privacy"},
        {"doc_id": "policy-eval", "title": "GenAI evaluation", "topic": "evaluation"},
        {"doc_id": "policy-support", "title": "Support readiness", "topic": "support"},
        {"doc_id": "policy-billing", "title": "Billing and invoices", "topic": "billing"},
        {"doc_id": "policy-observability", "title": "Production observability", "topic": "operations"},
        {"doc_id": "policy-incident", "title": "Incident response", "topic": "operations"},
        {"doc_id": "policy-onboarding", "title": "Customer onboarding", "topic": "customer-success"},
        {"doc_id": "policy-security-review", "title": "Security review", "topic": "security"},
        {"doc_id": "policy-latency", "title": "Latency and performance", "topic": "performance"},
        {"doc_id": "policy-feedback", "title": "User feedback loops", "topic": "product"},
        {"doc_id": "policy-release", "title": "Release approvals", "topic": "release"},
    ]
)

document_catalog_df

In [ ]:
query_specs = [
    {"query_id": "q_auth_sso", "query": "What should enterprise authentication include before launch?", "relevant": ["policy-auth"]},
    {"query_id": "q_privacy_boundary", "query": "Can customer documents leave the tenant boundary?", "relevant": ["policy-privacy"]},
    {"query_id": "q_eval_launch", "query": "What should we check before launching a mocked GenAI feature?", "relevant": ["policy-eval", "policy-support"]},
    {"query_id": "q_billing_invoice", "query": "How should invoice disputes be handled?", "relevant": ["policy-billing"]},
    {"query_id": "q_enterprise_rollout", "query": "What should be ready for an enterprise rollout?", "relevant": ["policy-auth", "policy-support", "policy-release"]},
    {"query_id": "q_data_residency", "query": "What protects private customer content in a hosted tenant?", "relevant": ["policy-privacy", "policy-security-review"]},
    {"query_id": "q_support_escalation", "query": "Who needs to be ready before support escalation starts?", "relevant": ["policy-support", "policy-incident"]},
    {"query_id": "q_rollback_launch", "query": "What launch plan should include rollback details?", "relevant": ["policy-auth", "policy-release"]},
    {"query_id": "q_citation_eval", "query": "How do we verify generated answers cite the right evidence?", "relevant": ["policy-eval"]},
    {"query_id": "q_external_service", "query": "Can we send private content to a third-party service?", "relevant": ["policy-privacy", "policy-security-review"]},
    {"query_id": "q_known_issues", "query": "What should customer-facing teams receive before rollout?", "relevant": ["policy-support", "policy-release"]},
    {"query_id": "q_failure_cases", "query": "How should we review failure cases for the assistant?", "relevant": ["policy-eval", "policy-feedback"]},
    {"query_id": "q_saml_oidc", "query": "Which identity protocols do enterprise customers require?", "relevant": ["policy-auth"]},
    {"query_id": "q_tenant_isolation", "query": "How should tenant isolation affect retrieval?", "relevant": ["policy-privacy", "policy-security-review"]},
    {"query_id": "q_api_latency", "query": "What should we monitor if answer generation gets slow?", "relevant": ["policy-latency", "policy-observability"]},
    {"query_id": "q_incident_comms", "query": "What happens when a production issue affects customers?", "relevant": ["policy-incident", "policy-support"]},
    {"query_id": "q_onboarding_setup", "query": "What should happen during customer onboarding?", "relevant": ["policy-onboarding", "policy-auth"]},
    {"query_id": "q_feedback_loop", "query": "How should product teams learn from bad assistant answers?", "relevant": ["policy-feedback", "policy-eval"]},
    {"query_id": "q_security_approval", "query": "Who should approve risky integrations before launch?", "relevant": ["policy-security-review", "policy-release"]},
    {"query_id": "q_release_gate", "query": "What should block a production release?", "relevant": ["policy-release", "policy-eval", "policy-security-review"]},
]

queries_df = pd.DataFrame(
    [{"query_id": item["query_id"], "query": item["query"]} for item in query_specs]
)

relevance_df = pd.DataFrame(
    [
        {"query_id": item["query_id"], "doc_id": doc_id, "is_relevant": True}
        for item in query_specs
        for doc_id in item["relevant"]
    ]
)

queries_df.head(), relevance_df.head()

In [ ]:
retrieval_runs = {
    "q_auth_sso": [
        ["policy-auth", "policy-support", "policy-security-review", "policy-release", "policy-privacy"],
        ["policy-auth", "policy-eval", "policy-support", "policy-release", "policy-latency"],
        ["policy-support", "policy-auth", "policy-privacy", "policy-onboarding", "policy-release"],
    ],
    "q_privacy_boundary": [
        ["policy-privacy", "policy-security-review", "policy-auth", "policy-eval", "policy-support"],
        ["policy-privacy", "policy-security-review", "policy-support", "policy-auth", "policy-release"],
        ["policy-privacy", "policy-security-review", "policy-eval", "policy-incident", "policy-support"],
    ],
    "q_eval_launch": [
        ["policy-eval", "policy-support", "policy-feedback", "policy-release", "policy-privacy"],
        ["policy-support", "policy-eval", "policy-release", "policy-auth", "policy-feedback"],
        ["policy-eval", "policy-privacy", "policy-feedback", "policy-support", "policy-release"],
    ],
    "q_billing_invoice": [
        ["policy-support", "policy-auth", "policy-privacy", "policy-onboarding", "policy-incident"],
        ["policy-privacy", "policy-support", "policy-eval", "policy-feedback", "policy-release"],
        ["policy-auth", "policy-eval", "policy-support", "policy-latency", "policy-onboarding"],
    ],
    "q_enterprise_rollout": [
        ["policy-auth", "policy-support", "policy-release", "policy-eval", "policy-onboarding"],
        ["policy-privacy", "policy-auth", "policy-support", "policy-release", "policy-security-review"],
        ["policy-auth", "policy-eval", "policy-privacy", "policy-support", "policy-release"],
    ],
    "q_data_residency": [
        ["policy-privacy", "policy-security-review", "policy-auth", "policy-observability", "policy-support"],
        ["policy-security-review", "policy-privacy", "policy-release", "policy-incident", "policy-auth"],
        ["policy-privacy", "policy-auth", "policy-onboarding", "policy-support", "policy-security-review"],
    ],
    "q_support_escalation": [
        ["policy-support", "policy-incident", "policy-observability", "policy-feedback", "policy-release"],
        ["policy-incident", "policy-support", "policy-privacy", "policy-observability", "policy-auth"],
        ["policy-support", "policy-release", "policy-feedback", "policy-incident", "policy-onboarding"],
    ],
    "q_rollback_launch": [
        ["policy-release", "policy-auth", "policy-support", "policy-incident", "policy-eval"],
        ["policy-auth", "policy-release", "policy-security-review", "policy-support", "policy-observability"],
        ["policy-support", "policy-release", "policy-auth", "policy-feedback", "policy-eval"],
    ],
    "q_citation_eval": [
        ["policy-eval", "policy-feedback", "policy-support", "policy-release", "policy-privacy"],
        ["policy-eval", "policy-privacy", "policy-feedback", "policy-auth", "policy-support"],
        ["policy-feedback", "policy-eval", "policy-support", "policy-latency", "policy-release"],
    ],
    "q_external_service": [
        ["policy-security-review", "policy-privacy", "policy-release", "policy-auth", "policy-incident"],
        ["policy-privacy", "policy-security-review", "policy-support", "policy-observability", "policy-release"],
        ["policy-release", "policy-privacy", "policy-security-review", "policy-auth", "policy-feedback"],
    ],
    "q_known_issues": [
        ["policy-support", "policy-release", "policy-feedback", "policy-onboarding", "policy-incident"],
        ["policy-feedback", "policy-support", "policy-release", "policy-eval", "policy-auth"],
        ["policy-support", "policy-incident", "policy-onboarding", "policy-release", "policy-privacy"],
    ],
    "q_failure_cases": [
        ["policy-eval", "policy-feedback", "policy-support", "policy-observability", "policy-release"],
        ["policy-feedback", "policy-eval", "policy-latency", "policy-support", "policy-privacy"],
        ["policy-eval", "policy-release", "policy-security-review", "policy-feedback", "policy-support"],
    ],
    "q_saml_oidc": [
        ["policy-auth", "policy-security-review", "policy-onboarding", "policy-release", "policy-privacy"],
        ["policy-auth", "policy-support", "policy-security-review", "policy-eval", "policy-release"],
        ["policy-auth", "policy-privacy", "policy-onboarding", "policy-latency", "policy-support"],
    ],
    "q_tenant_isolation": [
        ["policy-privacy", "policy-security-review", "policy-auth", "policy-observability", "policy-incident"],
        ["policy-security-review", "policy-privacy", "policy-release", "policy-auth", "policy-support"],
        ["policy-privacy", "policy-onboarding", "policy-security-review", "policy-feedback", "policy-auth"],
    ],
    "q_api_latency": [
        ["policy-latency", "policy-observability", "policy-incident", "policy-support", "policy-release"],
        ["policy-observability", "policy-latency", "policy-feedback", "policy-eval", "policy-support"],
        ["policy-latency", "policy-support", "policy-observability", "policy-incident", "policy-privacy"],
    ],
    "q_incident_comms": [
        ["policy-incident", "policy-support", "policy-observability", "policy-release", "policy-feedback"],
        ["policy-support", "policy-incident", "policy-feedback", "policy-onboarding", "policy-release"],
        ["policy-observability", "policy-incident", "policy-support", "policy-latency", "policy-privacy"],
    ],
    "q_onboarding_setup": [
        ["policy-onboarding", "policy-auth", "policy-support", "policy-release", "policy-privacy"],
        ["policy-auth", "policy-onboarding", "policy-security-review", "policy-support", "policy-feedback"],
        ["policy-onboarding", "policy-support", "policy-auth", "policy-incident", "policy-release"],
    ],
    "q_feedback_loop": [
        ["policy-feedback", "policy-eval", "policy-support", "policy-observability", "policy-release"],
        ["policy-eval", "policy-feedback", "policy-privacy", "policy-support", "policy-latency"],
        ["policy-feedback", "policy-support", "policy-eval", "policy-incident", "policy-release"],
    ],
    "q_security_approval": [
        ["policy-security-review", "policy-release", "policy-privacy", "policy-auth", "policy-incident"],
        ["policy-release", "policy-security-review", "policy-observability", "policy-privacy", "policy-support"],
        ["policy-security-review", "policy-auth", "policy-release", "policy-onboarding", "policy-eval"],
    ],
    "q_release_gate": [
        ["policy-release", "policy-eval", "policy-security-review", "policy-support", "policy-incident"],
        ["policy-eval", "policy-release", "policy-feedback", "policy-security-review", "policy-privacy"],
        ["policy-security-review", "policy-release", "policy-auth", "policy-eval", "policy-observability"],
    ],
}

rows = []
for query_id, runs in retrieval_runs.items():
    for run_index, retrieved_doc_ids in enumerate(runs, start=1):
        for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
            rows.append(
                {
                    "query_id": query_id,
                    "run_id": f"run_{run_index}",
                    "rank": rank,
                    "doc_id": doc_id,
                    "score": round(1.0 - (rank * 0.08) - (run_index * 0.01), 3),
                }
            )

retrieval_df = pd.DataFrame(rows).merge(queries_df, on="query_id", how="left")
retrieval_df = retrieval_df.merge(document_catalog_df, on="doc_id", how="left")

retrieval_df.head(15)

In [ ]:
print(f"queries: {queries_df['query_id'].nunique()}")
print(f"retrieval rows: {len(retrieval_df)}")
print(f"runs per query: {retrieval_df.groupby('query_id')['run_id'].nunique().unique().tolist()}")

retrieval_df.sample(10, random_state=7)

## Task

Use the DataFrames above to evaluate the retrieval outputs. Decide what metrics or summaries are useful, implement them below, and explain what you learn.

Some things to keep in mind without treating them as a checklist: the pipeline may only send the top few chunks to the prompt, some queries have more than one relevant document, and each query was retrieved three times.

In [ ]:
# Implement your retrieval evaluation here.
# Feel free to create helper functions, merge DataFrames, group by query/run,
# or use any library you think is appropriate.


## Reasoning Prompt

After you compute your evaluation, explain:

- Which queries look healthy?
- Which query or behavior is the biggest product risk?
- Do any queries behave differently across repeated runs?
- What would you change in the retrieval pipeline or evaluation dataset next?